# Notebook 19 — is the perception signal the model's, or the extractor's?

The one open confound the perception arm cannot resolve by rescoring.
`extract_final_answer` has four tiers, and on **153/300 items it fires a
different tier across the five samples**. Mean entropy rises monotonically
with that count (0.685 → 1.031 → 1.243 at 1/2/3 tiers). So an unknown share of
what the arm scores as *model* uncertainty may be the extractor changing its
mind about which line of an unchanged derivation is the answer.

**No rescoring can separate the two** — every rule reads the same ambiguous
text. Notebook 17 measured the confound; it cannot remove it.

This run removes it. The transcription prompt additionally asks for the final
answer in `\boxed{}`, so `extract_final_answer`'s boxed tier fires first and
deterministically. **The entropy that remains is the model's.**

Same 300 images, same seed, same K=5, same model (Qwen2.5-VL-3B) as the
reference run — only the prompt changes.

**Registered before running** (`pilot.rescore.BOXED_PREREGISTRATION`):

| check | bar | why |
|---|---|---|
| `\boxed{}` compliance | ≥ 80% of samples | below this the run measures instruction-following, not uncertainty |
| multi-tier items | ≤ 10% | if extraction still varies, the manipulation did not work |
| perception AUROC | ≥ 0.75, CI excluding chance | the signal survives deterministic extraction |

Verdicts come from `pilot.rescore.classify_boxed_result`, so the reported
label is the tested one. **`signal_was_extractor` is a real possible outcome
and must be reported if it happens.**

The prompt is additive and repeats that the task is still OCR. The confound to
avoid is the model switching from *transcribing* to *solving*, which would
silently change what is measured — the gate warns if accuracy moves >10 points.

In [ ]:
# Install. Qwen2.5-VL needs a current transformers; nothing exotic.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes

In [ ]:
# Auth & code access. Identical to notebooks 12-15, including the
# sys.modules purge that makes a re-clone actually take effect.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Stored HF token does not start with 'hf_'; set RESET_TOKENS = True.")
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for _n in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_n]

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print("pilot imported from:", os.path.dirname(pilot.__file__))

In [ ]:
# Model load. Qwen2.5-VL-3B -- the SAME model as the reference n=300 run.
# Changing the model and the prompt at once would make this uninterpretable.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
QUANTIZED = False          # bf16, matching the reference run

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()
print(f"loaded {MODEL_ID}  quantized={QUANTIZED}  dtype={model.dtype}")

In [ ]:
# Sample. ALWAYS drawn at n=300 -- the same call every reference run used, so
# the comparison is apples-to-apples -- then truncated to PROCESS_N for the
# gate. Because the draw is identical, the gate items are a prefix of the full
# run and the checkpoint carries straight over. Set PROCESS_N = 300 and re-run
# the generation cell to continue; the first 50 are not regenerated.
import logging

import pilot.data

logging.basicConfig(level=logging.INFO)

SAMPLE_N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5
PROCESS_N = 300          # <-- the gate. Raise to 300 only after the gate passes.

full_sample = pilot.data.load_fermat_balanced(
    n=SAMPLE_N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC)
sample = full_sample.select(range(PROCESS_N))
print(f"drawn {len(full_sample)} items, processing the first {len(sample)}")
print(f"has_error in this slice: {sum(bool(x) for x in sample['has_error'])}/{len(sample)}")

In [ ]:
# Adapter + pre-flight. Qwen2.5-VL takes the standard system+user shape, so
# this is pilot.prompts.build_transcription_messages_boxed unchanged.
import pilot.prompts


def qwen_inputs(messages):
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    images = [c["image"] for m in messages for c in m["content"]
              if c.get("type") == "image"]
    return processor(text=[text], images=images, return_tensors="pt").to(model.device)


# Pre-flight on one item: confirms the adapter works AND that the model
# actually complies with \boxed{} before a session is spent on it.
_probe = qwen_inputs(pilot.prompts.build_transcription_messages_boxed(
    full_sample[0]["image"]))
with torch.no_grad():
    _out = model.generate(**_probe, max_new_tokens=256, do_sample=False)
_text = processor.batch_decode(_out[:, _probe["input_ids"].shape[1]:],
                               skip_special_tokens=True)[0]
print(_text[:700])
print("\n" + "=" * 70)
print("contains \\boxed{}:", "\\boxed{" in _text)
print("If this is False on a few probes, stop -- the gate will fail anyway.")

In [ ]:
# Generation: K=5, transcription ONLY (no grading arm in this run).
# Batch-backoff ladder and per-item Drive checkpoint, both carried over from
# notebook 16 unchanged. The checkpoint is keyed by SAMPLE_N (300), NOT by
# PROCESS_N, so raising PROCESS_N resumes rather than restarts.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_TRANSCRIPTION = 5
TEMP = 0.7
_LADDER = [5, 2, 1]
_state = {"i": 0}
META = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _batch(messages, n, temperature):
    inputs = qwen_inputs(messages)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=True,
                             temperature=temperature, num_return_sequences=n)
    texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True,
                                   clean_up_tokenization_spaces=False)
    del out, inputs
    gc.collect(); torch.cuda.empty_cache()
    return texts


def generate_k(messages, n, temperature):
    texts, last = [], None
    while len(texts) < n:
        size = min(_LADDER[_state["i"]], n - len(texts))
        for attempt in range(3):
            try:
                texts += _batch(messages, size, temperature)
                last = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if _state["i"] + 1 < len(_LADDER):
                    _state["i"] += 1
                    print(f"  OOM at batch {size}; dropping to {_LADDER[_state['i']]}",
                          flush=True)
                    size = min(_LADDER[_state["i"]], n - len(texts))
                    continue
                raise
            except INFRA as exc:
                last = exc
                gc.collect(); torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last is not None:
            raise last
    return texts


CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
slug = MODEL_ID.split("/")[-1]
ckpt = (f"{CHECKPOINT_DIR}/boxed_perc_{slug}_n{SAMPLE_N}_seed{SEED}"
        f"_k{K_TRANSCRIPTION}{'_4bit' if QUANTIZED else ''}.jsonl")

raw_results = []
if os.path.exists(ckpt):
    with open(ckpt) as f:
        raw_results = [json.loads(l) for l in f if l.strip()]
    valid = []
    for idx, e in enumerate(raw_results):
        if idx >= len(full_sample):
            break
        it = full_sample[idx]
        if not all(e["item"].get(k) == it[k] for k in META):
            print(f"checkpoint item {idx+1} mismatch; resuming there."); break
        if len(e.get("transcription_samples_raw", [])) != K_TRANSCRIPTION:
            break
        valid.append(e)
    if len(valid) != len(raw_results):
        with open(ckpt, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
    raw_results = valid
    print(f"resuming from {len(raw_results)} completed items")

todo = [i for i in range(PROCESS_N) if i >= len(raw_results)]
if not todo:
    print(f"first {PROCESS_N} items already done.")
else:
    print(f"generating items {todo[0]+1}..{todo[-1]+1}", flush=True)
    with tqdm(total=len(todo) * K_TRANSCRIPTION, desc="boxed", unit="sample") as pbar:
        for idx in todo:
            item = full_sample[idx]
            t0 = time.time()
            tr = generate_k(
                pilot.prompts.build_transcription_messages_boxed(item["image"]),
                K_TRANSCRIPTION, TEMP)
            pbar.update(K_TRANSCRIPTION)
            entry = {"item": {k: item[k] for k in META},
                     "transcription_samples_raw": tr,
                     "quantized": QUANTIZED, "elapsed_seconds": time.time() - t0}
            raw_results.append(entry)
            with open(ckpt, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n"); f.flush()
            print(f"  item {idx+1}/{PROCESS_N}: {time.time()-t0:.1f}s", flush=True)

print(f"raw_results: {len(raw_results)} items")

In [ ]:
# THE GATE. Scores what exists and applies the three PRE-REGISTERED checks
# from pilot.rescore.BOXED_PREREGISTRATION. The verdict comes from
# classify_boxed_result, so the label reported is the one that was tested.
import math

import pandas as pd

import pilot.rescore
from pilot.plotting import bootstrap_auroc_ci

rows = []
for e in raw_results:
    tr = e["transcription_samples_raw"]
    scored = pilot.rescore.score_item(tr, e["item"]["pert_a"], "strict_v1")
    comp = pilot.rescore.boxed_compliance(tr)
    tiers = pilot.rescore.tier_instability(tr)
    rows.append({**{k: e["item"][k] for k in META},
                 "perception_entropy": scored["perception_entropy"],
                 "transcription_correct": scored["transcription_correct"],
                 "n_transcription_parse_failures":
                     scored["n_transcription_parse_failures"],
                 "frac_boxed": comp["frac_boxed"],
                 "n_distinct_tiers": tiers["n_distinct_tiers"],
                 "all_transcription_samples_raw": tr,
                 "quantized": e["quantized"], "model_id": MODEL_ID,
                 "k_transcription": K_TRANSCRIPTION})
scored_df = pd.DataFrame(rows)

compliance = scored_df["frac_boxed"].mean()
multi_tier = (scored_df["n_distinct_tiers"] > 1).mean()
acc = scored_df["transcription_correct"].astype(bool).mean()

print(f"n = {len(scored_df)}")
print(f"  \\boxed{{}} compliance      : {compliance:.1%}   "
      f"(bar {pilot.rescore.BOXED_PREREGISTRATION['min_boxed_compliance']:.0%})")
print(f"  items using >1 tier       : {multi_tier:.1%}   "
      f"(bar <={pilot.rescore.BOXED_PREREGISTRATION['max_multi_tier_frac']:.0%}"
      f"; the reference run was 51.0%)")
print(f"  transcription accuracy    : {acc:.1%}   (reference run: 47.0%)")

# Deliberately no AUROC below the full sample: fewer items cannot reach the
# registered 30-item minority minimum, and a number printed here would only
# get quoted. Same discipline as notebook 16's gate.
if len(scored_df) < 300:
    print(f"\nPROCESS_N={len(scored_df)} < 300 -- no AUROC printed by design. "
          "Raise PROCESS_N and re-run the generation cell.")
else:
    ci = bootstrap_auroc_ci(scored_df, "perception_entropy",
                            "transcription_correct", n_boot=10000, seed=0)
    verdict = pilot.rescore.classify_boxed_result(compliance, ci, multi_tier)
    print(f"  perception AUROC          : {ci['auroc']:.3f} "
          f"[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]   (reference: 0.835)")
    print(f"\n  REGISTERED VERDICT: {verdict}")
    print({
        "gated_low_compliance":
            "  -> the model ignored \\boxed{}; this run cannot answer the question.",
        "manipulation_failed":
            "  -> it complied but extraction still varies; the experiment did not work.",
        "signal_is_not_extractor":
            "  -> the AUROC holds under DETERMINISTIC extraction. The perception\n"
            "     signal is the model's uncertainty, not extractor instability.",
        "signal_was_extractor":
            "  -> it collapses to chance. A substantial part of the original signal\n"
            "     was extractor instability. REPORT THIS -- it is a real outcome.",
        "inconclusive":
            "  -> between the two at this n.",
    }[verdict])

# The confound the prompt was written to avoid: if accuracy moved a lot, the
# model may have started SOLVING rather than transcribing, which would change
# what is being measured regardless of what the AUROC says.
if abs(acc - 0.470) > 0.10:
    print(f"\n  WARNING: accuracy moved {acc - 0.470:+.1%} vs the reference run. "
          "Check a few samples for the model solving instead of transcribing "
          "before interpreting anything above.")

In [ ]:
# Save. Drive first, then repo + push -- Drive is the source of truth because
# pushes from Colab 403 routinely here (see notebook 15).
import os
import subprocess
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
tag = "gate" if PROCESS_N < 300 else "full"
csv_name = (f"pixtral_perception_{tag}_n{PROCESS_N}_"
            f"{'4bit_' if QUANTIZED else ''}pixtral-12b_{timestamp}.csv")

drive_results = f"{PROJECT_DIR}/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
df.to_csv(f"repo/results/{csv_name}", index=False)
print(f"Wrote repo/results/{csv_name} ({len(df)} rows)")

_REDACT = []


def git(*args):
    r = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for s in _REDACT:
        if s:
            out = out.replace(s, "***")
    if r.returncode != 0 and out.strip():
        print(out.strip())
    return r


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
if git("commit", "-m", f"Add boxed-12B perception results: {csv_name}").returncode != 0:
    print("git commit failed -- CSV is safe on Drive.")

tok = (globals().get("GH_TOKEN") or "").strip()
_REDACT.append(tok)
pushed = False
if tok:
    url = REPO_URL.replace("https://", f"https://{tok}@")
    if git("fetch", url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
    pushed = git("push", url, "HEAD:main").returncode == 0
print("Pushed." if pushed else
      f"Push failed or skipped -- the CSV is on Drive at {drive_results}/{csv_name}, "
      "nothing is lost.")

## What to do with the result

- **`signal_is_not_extractor`** — the strongest outcome available. It converts
  the Limitations sentence from *"part of the perception signal may be
  extractor instability"* into *"it is not"*, on a direct manipulation rather
  than an argument. Report the AUROC next to the reference 0.835.
- **`signal_was_extractor`** — report it. It would be the single most
  important negative result in the project, and it is exactly why the bar was
  registered in advance.
- **`gated_low_compliance`** — a 3B model may simply not follow the format.
  Not a failure of the idea; try the same prompt on Qwen2.5-VL-7B before
  concluding anything.
- Either way the CSV is Drive-only, as always. Snapshot it into
  `reference/` only if it becomes claim-bearing.